<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/voice_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ⚙️ BƯỚC 1: CÀI ĐẶT HỆ THỐNG ALL-IN-ONE
import os
from IPython.display import Audio, display, clear_output

print("⏳ Đang cài đặt thư viện lõi (Chỉ mất 1-2 phút)...")
os.system('pip install -U qwen-tts huggingface_hub pydub')
os.system('apt-get install -y ffmpeg sox libsox-fmt-all')
clear_output()

from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import gc

torch.backends.cudnn.benchmark = True
current_model = None
current_model_name = None

# Hàm tải mô hình thông minh (Chống sập RAM Colab)
def load_qwen_model(model_id):
    global current_model, current_model_name

    # Nếu model đang dùng giống model yêu cầu -> Dùng luôn cho lẹ
    if current_model_name == model_id and current_model is not None:
        return current_model

    # Nếu đang dùng model khác -> Xóa model cũ đi để giải phóng RAM
    if current_model is not None:
        print("🧹 Đang dọn dẹp bộ nhớ RAM...")
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model: {model_id.split('/')[-1]}...")
    current_model = Qwen3TTSModel.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="cuda:0",
        attn_implementation="sdpa"
    )
    current_model_name = model_id
    print("✅ Đã tải xong Model!")
    return current_model

print("✅ HỆ THỐNG ALL-IN-ONE ĐÃ SẴN SÀNG!")


********
********
 
✅ HỆ THỐNG ALL-IN-ONE ĐÃ SẴN SÀNG!


In [3]:
# @title 🎨 CHỨC NĂNG 1: VOICE DESIGN (Tạo giọng mới)

# MÔ TẢ GIỌNG BẠN MUỐN TẠO (Bằng tiếng Anh):
mo_ta_giong = """

A old female voice, sad.

"""

# CÂU NÓI ĐỂ AI ĐỌC THỬ:
cau_noi_thu = """

Once upon a time, in a magical forest, there lived a tiny fairy.

"""

# ------------------------------------------------------------------
model = load_qwen_model("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")

print("🎨 Đang thiết kế giọng theo yêu cầu của bạn...")
with torch.inference_mode():
    w, sr = model.generate_voice_design(text=cau_noi_thu, instruct=mo_ta_giong)

ten_file = "1_VoiceDesign_Output.wav"
sf.write(ten_file, w[0], sr)

clear_output()
print("🎉 Đã tạo giọng thành công!")
display(Audio(ten_file))

🎉 Đã tạo giọng thành công!


In [5]:
# @title 🗣️ CHỨC NĂNG 2: CLONE GIỌNG ĐIỆN ẢNH (Ngắt nghỉ chuyên sâu)

import os
import re
import torch
import soundfile as sf
from pydub import AudioSegment
from IPython.display import Audio, display, clear_output

# TÊN FILE GIỌNG MẪU BẠN ĐÃ TẢI LÊN COLAB (Cột bên trái 📁):
file_giong_mau = "yo.wav"

# LỜI THOẠI MÀ NGƯỜI TRONG FILE MẪU ĐANG NÓI:
loi_thoai_giong_mau = """
Wait, have you ever noticed that time feels faster as we get older? It's kind of scary, right? But actually, there is a hidden logic behind it.
"""

# VĂN BẢN BẠN MUỐN AI ĐỌC (Cứ trình bày, cách dòng tự nhiên như đang gõ Word):
van_ban_can_doc = """
You have a supercomputer in your pocket.
It synchronizes with atomic clocks. It tracks the global financial markets in real-time. It connects you to every piece of human knowledge ever recorded.

And yet... somewhere in the back of your mind, there is a quiet, persistent desire to spend ten, twenty, or fifty thousand dollars... on a heavy disk of mechanical steel to strap to your wrist.

You don't buy a Rolex to tell time.
You know this; the logic is undeniable.

But you tell yourself it’s an investment. An heirloom. A milestone of your success.
"""

# ------------------------------------------------------------------
if not os.path.exists(file_giong_mau):
    print(f"⚠️ LỖI: Chưa thấy file '{file_giong_mau}'. Vui lòng tải lên!")
else:
    model = load_qwen_model("Qwen/Qwen3-TTS-12Hz-1.7B-Base")

    print("⚡️ Đang học đa chiều từ giọng mẫu...")
    clone_prompt = model.create_voice_clone_prompt(
        ref_audio=file_giong_mau,
        ref_text=loi_thoai_giong_mau.strip(),
        x_vector_only_mode=False
    )

    # --- THIẾT LẬP THỜI GIAN NGẮT NGHỈ CHUYÊN SÂU ---
    # 1000 mili-giây = 1 giây
    nghi_ngan = AudioSegment.silent(duration=600)  # 0.6s: Nghỉ lấy hơi sau câu bình thường (Dấu chấm, chấm phẩy, xuống dòng)
    nghi_dai  = AudioSegment.silent(duration=1500) # 1.5s: Nghỉ kịch tính khi chuyển đoạn (Sang dòng trống)

    final_audio = AudioSegment.silent(duration=300) # Mở đầu nghỉ 0.3s

    # Xử lý văn bản: Tách thành các ĐOẠN (dựa vào dòng trống)
    raw_text = van_ban_can_doc.replace('\r', '')
    paragraphs = re.split(r'\n\s*\n', raw_text)

    print("🚀 BẮT ĐẦU THU ÂM VỚI NHỊP ĐIỆU CHUYÊN GIA...")

    line_count = 0
    # Tính tổng số câu để hiển thị tiến độ
    total_sentences = sum([len(re.split(r'(?<=[.?!;:\n])\s+', p.strip())) for p in paragraphs if p.strip()])

    for p in paragraphs:
        p = p.strip()
        if not p: continue

        # Tách các CÂU trong đoạn (dựa vào dấu ., ?, !, ;, : và xuống dòng)
        # Việc này giúp giữ nguyên dấu phẩy (,) trong câu để AI đọc nối chữ cho mượt
        sentences = re.split(r'(?<=[.?!;:\n])\s+', p)

        for s in sentences:
            s = s.strip()
            if not s: continue

            line_count += 1
            print(f"🎙️ Đang đọc [{line_count}/{total_sentences}]: {s[:60]}...")

            with torch.inference_mode():
                w, sr = model.generate_voice_clone(text=s, voice_clone_prompt=clone_prompt)

            sf.write("temp_line.wav", w[0], sr)
            # Ghép âm thanh + khoảng nghỉ NGẮN
            final_audio += AudioSegment.from_wav("temp_line.wav") + nghi_ngan

            if os.path.exists("temp_line.wav"): os.remove("temp_line.wav")

        # Hết 1 đoạn (gặp dòng trống) -> Thêm khoảng nghỉ DÀI để người nghe ngẫm nghĩ
        print("⏳ (Chuyển ý -> Nghỉ sâu 1.5 giây)")
        final_audio += nghi_dai

    # Xuất file mp3
    ten_file = "Kien_Thuc_Podcast.mp3"
    final_audio.export(ten_file, format="mp3")

    clear_output()
    print("🎉 HOÀN TẤT! Hãy nghe thử nhịp điệu điện ảnh nhé:")
    display(Audio(ten_file))

🎉 HOÀN TẤT! Hãy nghe thử nhịp điệu điện ảnh nhé:
